# diag1 — BiMamba ≡ 순방향 스택 + 위치별 상수표 `b_k`

diag0 에서 역방향 브랜치의 쿼리 위치 출력이 **관측과 무관한 상수**임을 확정했다
(K=50/100/150 전부 `max|Δ| = 0`, 역방향 크기는 순방향의 0.84배).

상수라면 **한 번 뽑아서 표로 저장**할 수 있다. 그러면 역방향 Mamba-2 스캔을
통째로 지워도 출력이 똑같아야 한다. 근사가 아니라 **항등식**이다.

```
원본     out_k = LayerNorm( 0.5 · ( forward_k + backward_k ) )
                                              └── 항상 같은 값
컴파일   out_k = LayerNorm( 0.5 · ( forward_k + b_k ) )      ← 표에서 읽음
```

## 이 노트북이 하는 일

| | |
|---|---|
| 1 | `b_k` 추출 — forward pass 한 번, `(K, 512)` |
| 2 | 디코더 컴파일 — `backward_layers` 를 `b_k` lookup 으로 교체 |
| 3 | **항등식 검증** — 원본 vs 컴파일 `max\|Δ\|` |
| 4 | 파라미터·지연 비교 |
| 5 | `b_k` 구조 — 위치별 norm, 위치 간 코사인 유사도 |

**3번이 핵심이고 4·5번이 논문에 들어갈 재료다.**

검증을 두 배치에서 한다 — `b_k` 를 뽑은 배치와, **한 번도 안 쓴 배치**. 후자가
같아야 "그 배치에서만 맞는 값" 이 아니라는 게 보장된다.

5번은 "긴 스캔이 잃은 위치 정체성을 head 직전에 복원한다" 가설의 직접 증거다.
`b_k` 가 이웃끼리 비슷하고 멀수록 달라지면 매끄러운 위치 부호라는 뜻이다.

---

> 커널 얘기는 diag0 과 같다. `lerobot` 을 커널로 import 하지 않고,
> `mamba_ssm` 이 깔린 venv 로 subprocess 호출한 뒤 결과 json 만 읽는다.


## 0) 부팅

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

_h = Path.cwd()
REPO = next(c for c in (_h, *_h.parents)
            if (c / 'notebooks' / 'libero' / 'diag1_bk_identity.py').exists())
sys.path.insert(0, str(REPO / 'notebooks'))

try:
    import common_v23 as v23
    PYTHON = v23.PYTHON
except Exception as e:
    print('common_v23 import 실패, 기본값 사용:', e)
    PYTHON = os.environ.get('LEROBOT_PYTHON') or str(
        Path.home() / 'lerobot_project' / 'lerobot_env' / 'bin' / 'python')

SCRIPT = REPO / 'notebooks' / 'libero' / 'diag1_bk_identity.py'
SHARE = Path(os.environ.get('LEROBOT_OUTPUT',
                            Path.home() / 'lerobot_project' / 'outputs')) / 'final' / 'share' / 'diag1'
SHARE.mkdir(parents=True, exist_ok=True)

print('repo   :', REPO)
print('python :', PYTHON, '  (있음)' if Path(PYTHON).exists() else '  ← 없다!')
print('script :', SCRIPT, '  (있음)' if SCRIPT.exists() else '  ← 없다!')
print('share  :', SHARE)

## 1) 설정

In [ ]:
SEED   = 0
STEP   = 150_000
TASK   = 'libero_10'
BATCH  = 4
GPU    = '0'
STAMP  = time.strftime('%Y%m%d_%H%M')

ENV = dict(os.environ,
           PYTHONPATH=str(REPO / 'src'),
           HF_HUB_DISABLE_XET='1',
           MPLBACKEND='Agg',
           CUDA_VISIBLE_DEVICES=GPU)

def run_diag1(tags, json_path):
    cmd = [PYTHON, str(SCRIPT), '--tags', tags, '--seed', str(SEED), '--step', str(STEP),
           '--task', TASK, '--batch', str(BATCH),
           '--save-dir', str(SHARE), '--json', str(json_path)]
    print('$', ' '.join(cmd), '\n')
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=ENV, cwd=str(REPO))
    for line in p.stdout:
        print(line, end='')
    print(f'\n[exit {p.wait()}]')
    return json.loads(Path(json_path).read_text(encoding='utf-8')) if Path(json_path).exists() else {}

## 2) K=100 — 이 셀이 항등식의 답이다

In [ ]:
res100 = run_diag1('bimamba_pure', SHARE / f'diag1_k100_{STAMP}.json')

## 3) K 50 / 100 / 150 전부

In [ ]:
results = run_diag1('all', SHARE / f'diag1_all_{STAMP}.json')
print('\n받은 태그:', list(results))

## 4) 요약표

In [ ]:
import csv

K_OF = {'bimamba_pure_k50': 50, 'bimamba_pure': 100, 'bimamba_pure_k150': 150}
order = sorted(results.items(), key=lambda kv: K_OF.get(kv[0], 0))

hdr = (f"{'K':>5} {'판정':<20} {'max|Δ| (본 배치)':>18} {'max|Δ| (안 본 배치)':>20} "
       f"{'역방향 params':>14} {'b_k 표':>10} {'지연':>16}")
print(hdr); print('-' * len(hdr))
rows = []
for tag, r in order:
    if not r.get('ok') and 'identity_max_diff_seen' not in r:
        print(f"{K_OF.get(tag, 0):>5} {r.get('verdict', '?'):<20}  — 건너뜀")
        continue
    print(f"{r['K']:>5} {r['verdict']:<20} {r['identity_max_diff_seen']:18.3e} "
          f"{r['identity_max_diff_unseen']:20.3e} {r['n_params_backward']:14,d} "
          f"{r['n_params_table']:10,d} "
          f"{r['latency_orig_ms']:6.2f}→{r['latency_compiled_ms']:5.2f} ms")
    rows.append({k: r.get(k) for k in
                 ('K', 'tag', 'verdict', 'identity_max_diff_seen', 'identity_max_diff_unseen',
                  'identity_rel', 'n_params_backward', 'n_params_forward', 'n_params_table',
                  'latency_orig_ms', 'latency_compiled_ms', 'speedup',
                  'cos_first_last', 'cos_offdiag_mean')})

if rows:
    csv_path = SHARE / f'diag1_summary_{STAMP}.csv'
    with csv_path.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    print('\nsaved', csv_path)

## 5) `b_k` 구조 — 위치 부호인가?

왼쪽: 위치별 `||b_k||`. 오른쪽: 위치 간 코사인 유사도 행렬.

대각선 근처가 밝고 멀어질수록 어두우면 **매끄러운 위치 부호**다 —
"몇 번째 액션인지" 를 연속적으로 encode 하고 있다는 뜻.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

avail = [(tag, r) for tag, r in order if r.get('npz') and Path(r['npz']).exists()]
if not avail:
    print('npz 가 없다. 셀 3) 을 먼저 돌릴 것.')
else:
    fig, axes = plt.subplots(2, len(avail), figsize=(4.2 * len(avail), 7.4),
                             squeeze=False)
    for j, (tag, r) in enumerate(avail):
        z = np.load(r['npz'])
        b, cos = z['b'], z['cos']
        K = b.shape[0]

        ax = axes[0][j]
        ax.plot(range(1, K + 1), np.linalg.norm(b, axis=-1), lw=1.5)
        ax.set_title(f"K={r['K']}  ||b_k||"); ax.set_xlabel('위치 k'); ax.grid(alpha=.3)

        ax = axes[1][j]
        im = ax.imshow(cos, cmap='viridis', origin='lower', vmin=-1, vmax=1,
                       extent=[1, K, 1, K])
        ax.set_title(f"K={r['K']}  cos(b_i, b_j)")
        ax.set_xlabel('j'); ax.set_ylabel('i')
        fig.colorbar(im, ax=ax, fraction=.046)
    fig.tight_layout()
    png = SHARE / f'diag1_bk_structure_{STAMP}.png'
    fig.savefig(png, dpi=150, bbox_inches='tight')
    print('saved', png)
    plt.show()

## 6) 이웃 유사도 — 위치가 멀어질수록 얼마나 달라지나

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for tag, r in avail:
    z = np.load(r['npz']); cos = z['cos']; K = cos.shape[0]
    # 오프셋 d 에 대한 평균 유사도
    d = np.arange(1, K)
    m = [float(np.diagonal(cos, offset=int(o)).mean()) for o in d]
    ax.plot(d, m, lw=1.6, label=f"K={r['K']}")
ax.axhline(0, color='k', lw=.6)
ax.set_xlabel('위치 간 거리 d'); ax.set_ylabel('cos(b_i, b_{i+d}) 평균')
ax.set_title('b_k 는 거리가 멀수록 달라지는가')
ax.grid(alpha=.3); ax.legend()
fig.tight_layout()
png2 = SHARE / f'diag1_bk_decay_{STAMP}.png'
fig.savefig(png2, dpi=150, bbox_inches='tight')
print('saved', png2)
plt.show()

## 7) 결과 읽는 법

### 항등식 (셀 4)

**`max|Δ| = 0` 둘 다** → 확정. 출력이 비트 단위로 같다.
`BiMamba = 자기 순방향 스택 + 고정 b_k 표` 가 근사가 아니라 **항등식**이다.
논문에 "역방향 스캔은 추론에서 삭제 가능하다" 를 증명과 함께 쓸 수 있다.

**`1e-7` 수준** → 부동소수점 오차. 결론 같음.

**그보다 크면** → 컴파일 경로가 원본과 다르다. `use_action_self_attention`
이나 carry 설정을 확인해야 한다.

⚠️ **"안 본 배치" 가 핵심이다.** `b_k` 를 뽑은 배치에서만 같으면 의미가 없다.
두 열이 다 0 이어야 한다.

### 파라미터·지연 (셀 4)

역방향 Mamba-2 스택이 `b_k` 표(K×512)로 줄어든다. 지연은 디코더만 줄어드는
것이라 end-to-end 이득은 제한적이다 — ResNet-18 백본이 크다.
**측정된 숫자 그대로만 쓰고 부풀리지 않는다.**

### `b_k` 구조 (셀 5–6)

대각선 근처가 밝고 거리 `d` 가 커질수록 유사도가 떨어지면 **매끄러운 위치 부호**다.
"긴 스캔이 위치 정체성을 잃고 BiMamba 가 head 직전에 복원한다" 가설의 직접 증거가 된다.

반대로 코사인이 전부 1 에 가까우면 `b_k` 가 위치를 거의 구분하지 않는다는 뜻이고,
그러면 단순한 전역 편향이라 가설을 다시 세워야 한다.

산출물은 `outputs/final/share/diag1/` 에 json · csv · npz · png 로 남는다.